### Introduction
Description: A 16–base pair deletion in the first exon of the FRIGIDA gene (FRI, At4g00650) is found in Col-0 and several other rapid-cycling (early-flowering) accessions. This small deletion causes a frameshift that truncates the FRI protein, removing its C-terminal coiled-coil domain. The result is a nonfunctional FRI allele (often referred to as fri). Col-0’s reference genome carries this deletion, explaining why Col-0 is early-flowering (it lacks active FRI function).

In terms of reference genome, it's a insertion
| Chr | Start   | End    | Ref| Alt             |
|-----|---------|--------|----|-----------------|
| 4   | 269960  | 269960 | T  |TTCTTGTCCCTATGGTC|

In [1]:
import pysam
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForMaskedLM
import torch
import numpy as np

In [2]:
def load_model(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    model = AutoModelForMaskedLM.from_pretrained(model_name, trust_remote_code=True).eval()
    return tokenizer, model

In [3]:
genome_file = '../../../results/SV_effect/Arabidopsis_thaliana.TAIR10.dna.toplevel.noMtPt.fa'
fasta = pysam.FastaFile(genome_file)

#### Assuming alt_seq doesn't have the sequence

In [4]:
start = 269960
end = 269960
chrom = '4'
del_len = 16

mut_seq = fasta.fetch('4', start - 4096, start + 4096) #  269960 is the 1-based genome coordinates
ins = 'TTCTTGTCCCTATGGTC'

In [5]:
ref_seq = mut_seq
ref_seq = mut_seq[:4095] + 'TTCTTGTCCCTATGGTC' + mut_seq[4096:]
ref_seq = ref_seq[(del_len//2):-(del_len//2)]

In [6]:
len(ref_seq), len(mut_seq)

(8192, 8192)

In [7]:
tokenizer, model = load_model('kuleshov-group/compo-cad2-l48-d1536-dna-chtk-c8k-1t-v1-b2-lr4e4-NzqiLr')
model.to('cuda:0')

/home/jz963/miniconda3/envs/transformers/lib/python3.11/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


CaduceusForMaskedLM(
  (caduceus): Caduceus(
    (backbone): CaduceusMixerModel(
      (embeddings): CaduceusEmbeddings(
        (word_embeddings): RCPSEmbedding(
          (embedding): Embedding(8, 1536)
        )
      )
      (layers): ModuleList(
        (0-47): 48 x RCPSMambaBlock(
          (mixer): RCPSWrapper(
            (submodule): BiMambaWrapper(
              (mamba_fwd): Mamba2(
                (in_proj): Linear(in_features=1536, out_features=6320, bias=False)
                (conv1d): Conv1d(3200, 3200, kernel_size=(4,), stride=(1,), padding=(3,), groups=3200)
                (act): SiLU()
                (norm): RMSNorm()
                (out_proj): Linear(in_features=3072, out_features=1536, bias=False)
              )
              (mamba_rev): Mamba2(
                (in_proj): Linear(in_features=1536, out_features=6320, bias=False)
                (conv1d): Conv1d(3200, 3200, kernel_size=(4,), stride=(1,), padding=(3,), groups=3200)
                (act): SiLU()
   

In [8]:
inputs = tokenizer(
            [ref_seq, mut_seq],
            truncation=False,
            padding=False,
            return_tensors="pt",
            return_attention_mask=False,
            return_token_type_ids=False,)['input_ids'].to(model.device)

with torch.no_grad():
    outputs = model(inputs)

nucleotides = list('acgt')
logits = outputs.logits[..., [tokenizer.get_vocab()[nc] for nc in nucleotides]]
probs = torch.nn.functional.softmax(logits, dim=2).cpu().numpy()
probs.shape

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


(2, 8192, 4)

In [9]:
flank_len = (8192-del_len)//2
flank_len

4088

In [10]:
ref_prob_left = probs[0][0:flank_len][-20:]
mut_prob_left = probs[1][0:4096][-20:]

ref_prob_right = probs[0][-flank_len:][0:20]
mut_prob_right = probs[1][-4096:][0:20]

In [11]:
ref_prob = np.concatenate((ref_prob_left, ref_prob_right), axis = 0)
mut_prob = np.concatenate((mut_prob_left, mut_prob_right), axis = 0)
seq = ref_seq[0:flank_len][-20:] + ref_seq[-flank_len:][0:20]

In [12]:
ref_prob.shape, mut_prob.shape, len(seq)

((40, 4), (40, 4), 40)

In [13]:
scores = []
nucleotides = "ACGT"
for idx, nt in enumerate(seq):
    if nt in nucleotides:
        refProb = ref_prob[idx, nucleotides.index(nt)]
        mutProb = mut_prob[idx, nucleotides.index(nt)]
        scores.append(np.log(mutProb / refProb))
    else:
        scores.append(0)

In [14]:
np.mean(scores)

-0.15011911

## The quantile of simulated deletions  using flanking 20bp (+10/-10)

| Percentile | Value      |
|------------|------------|
| 0.1%       | -1.6905446 |
| 1%         | -1.1289620 |
| 10%        | -0.4855522 |
| 50%        | -0.0777035 |

## Conclusion
Unfortunately, this insertion doesn't fall into the top 10%